In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [4]:
from datasets import load_dataset

In [6]:
train = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
train

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
        num_rows: 2000
    })
})

In [7]:
test = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
test

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E'],
        num_rows: 500
    })
})

## Q1

In [8]:
ds = train.map(lambda x: {'combined_text': x['prompt'] + ' ' + x['A']})
print(len(ds['train'][51]['combined_text']))

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

614


## Q2

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
tokenizer

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [11]:
print(tokenizer.vocab_size)

30522


## Q3

In [12]:
print(tokenizer.sep_token_id)

102


## Q4

In [14]:
tokens = tokenizer(list(train['train']['prompt']), padding='max_length', truncation=True, max_length=128, return_tensors='pt')

In [15]:
print(tokens['input_ids'].shape)

torch.Size([2000, 128])


**BERT/RoBERTa Architecture & Attention Mechanisms**

## Q5

In [16]:
768/12

64.0

## Q6

In [17]:
import torch
from transformers import AutoModel

In [18]:
model = AutoModel.from_pretrained('bert-base-uncased')

prompt = list(train['train']['prompt'])[0]
inputs = tokenizer(prompt, return_tensors='pt')

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
with torch.no_grad():
    output = model(**inputs)
print(output.last_hidden_state.shape)

torch.Size([1, 31, 768])


## Q7

In [20]:
cls_vector = output.last_hidden_state[0, 0, :]
print(round(cls_vector[:5].sum().item(), 4))

-1.2001


## Q8

In [21]:
model2 = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)

text = "Light-ion fusion is a technique."
inputs = tokenizer(text, return_tensors='pt')

print(tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']


In [22]:
with torch.no_grad():
    output2 = model2(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
fusion_idx = tokens.index('fusion')
attn = output2.attentions[-1][0, 0, 0, fusion_idx]

In [23]:
print(round(attn.item(), 4))

0.1025


**Context-Aware Embeddings**

## Q9

In [24]:
from sentence_transformers import SentenceTransformer, util

In [25]:
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [26]:
row = list(train['train'])[0]
emb_prompt = st_model.encode(row['prompt'])
emb_b = st_model.encode(row['B'])

In [27]:
print(round(util.cos_sim(emb_prompt, emb_b).item(), 4))

0.7658


## Q10

In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [30]:
def map3(actual, predicted):
    score, hits = 0, 0
    for i, p in enumerate(predicted[:3]):
        if p == actual and p not in predicted[:i]:
            hits += 1
            score += hits / (i + 1)
    return score

In [35]:
options = ['A', 'B', 'C', 'D', 'E']
train_data = list(train['train'])

In [39]:
combined = [r['prompt'] + ' ' + r['A'] + ' ' + r['B'] + ' ' + r['C'] + ' ' + r['D'] + ' ' + r['E'] for r in train_data]

tfidf = TfidfVectorizer(stop_words='english')
tfidf.fit(combined)

prompt_vecs = tfidf.transform([r['prompt'] for r in train_data])
option_vecs = {opt: tfidf.transform([r[opt] for r in train_data]) for opt in options}

scores = np.column_stack([
    cosine_similarity(prompt_vecs, option_vecs[opt]).diagonal() for opt in options
])

In [31]:
# Pipeline 2: MiniLM
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
prompt_embs = st_model.encode([r['prompt'] for r in train_data], batch_size=64, show_progress_bar=True)
option_embs = {opt: st_model.encode([r[opt] for r in train_data], batch_size=64, show_progress_bar=True) for opt in options}

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [33]:
minilm_preds = []
for i in range(len(train_data)):
    sims = [util.cos_sim(prompt_embs[i], option_embs[opt][i]).item() for opt in options]
    ranked = list(np.array(options)[np.argsort(sims)[::-1]])
    minilm_preds.append(ranked)

answers = [r['answer'] for r in train_data]
minilm_map3 = np.mean([map3(a, p) for a, p in zip(answers, minilm_preds)])
print(f"MiniLM MAP@3: {minilm_map3:.4f}")

MiniLM MAP@3: 0.4231


In [40]:
tfidf_preds = [list(np.array(options)[np.argsort(row)[::-1]]) for row in scores]

In [41]:
count = sum(
    1 for a, t, m in zip(answers, tfidf_preds, minilm_preds)
    if a not in t[:3] and a in m[:3]
)
print(f"MiniLM recovers but TF-IDF misses: {count}")

MiniLM recovers but TF-IDF misses: 613


**Zero-shot classification concepts**

## Q11

In [42]:
from transformers import pipeline

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [43]:
zsc = pipeline("zero-shot-classification")

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [44]:
row = list(train['train'])[1]
result = zsc(row['prompt'], candidate_labels=[row['A'], row['B'], row['C']])
print(round(result['scores'][0], 4))

0.4575


## Q12

In [45]:
result_multi = zsc(row['prompt'], candidate_labels=[row['A'], row['B'], row['C']], multi_label=True)

diff = abs(sum(result['scores']) - sum(result_multi['scores']))
print(round(diff, 4))

0.9995


## Q13

In [47]:
from transformers import AutoModelForSeq2SeqLM

In [48]:
tokenizer_flan = AutoTokenizer.from_pretrained("google/flan-t5-small")
model_flan = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [49]:
row = list(train['train'])[0]
text = f"Question: {row['prompt']}. Is the correct answer A: {row['A']} or B: {row['B']}? Answer with just the letter A or B."

inputs = tokenizer_flan(text, return_tensors='pt')
with torch.no_grad():
    outputs = model_flan.generate(**inputs, max_new_tokens=5)
print(tokenizer_flan.decode(outputs[0], skip_special_tokens=True))

B
